# Q-Learning in Gymnasium with Stable-Baselines3

This notebook demonstrates how to solve the `FrozenLake-v1` control problem with tabular Q-learning and with several neural-network-based agents from [Stable-Baselines3](https://stable-baselines3.readthedocs.io/). We begin with the classical update rule

$$
Q(s_t, a_t) \leftarrow (1-\alpha) Q(s_t, a_t) + \alpha \big[r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a')\big],
$$

where $\alpha$ is the learning rate and $\gamma$ the discount factor. The rest of the notebook compares exploration strategies, visualises learning dynamics, and finishes with a short discussion of the results.

## Experiment design

We will explore the following knobs and variations:

* **Exploration schedules** – a decaying $\varepsilon$-greedy policy for the tabular agent, together with the default linear exploration schedule used by DQN-like agents.
* **Learning rates and discounting** – the tabular agent uses a moderately small step size with $\gamma=0.99$; neural agents use Adam with the default SB3 schedules.
* **Neural Q-learning variants** – `DQN`, `QRDQN`, and `C51`, all of which are value-based algorithms that differ in how they represent the return distribution.

Throughout the notebook we log the episodic return, track Q-table convergence metrics, and compare the policies learned by the different agents.

In [ ]:
# If you are running this notebook on a fresh environment (e.g. Google Colab),
# uncomment the following line to install the required libraries.
# !pip install gymnasium stable-baselines3 matplotlib numpy pandas

In [ ]:
import math
import os
import random
from typing import Dict, List, Optional, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from gymnasium import spaces

from stable_baselines3 import C51, DQN, QRDQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

## Environment helper

The canonical tabular benchmarking task for Q-learning is Frozen Lake. The environment has 16 states and 4 discrete actions. We will use the slippery variant to keep the stochastic dynamics intact while limiting training time by disabling the *done on hole* mechanic.

In [ ]:
class DiscreteOneHotWrapper(gym.ObservationWrapper):
    """Convert discrete observations into a one-hot encoded vector."""

    def __init__(self, env: gym.Env):
        super().__init__(env)
        if not isinstance(env.observation_space, spaces.Discrete):
            raise TypeError("DiscreteOneHotWrapper expects a Discrete observation space")
        n = env.observation_space.n
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(n,), dtype=np.float32)

    def observation(self, observation: int) -> np.ndarray:
        one_hot = np.zeros(self.observation_space.shape, dtype=np.float32)
        one_hot[observation] = 1.0
        return one_hot


def make_frozenlake_env(seed: int = 0, monitor_dir: Optional[str] = None, one_hot: bool = False) -> gym.Env:
    env = gym.make("FrozenLake-v1", is_slippery=True)
    env.reset(seed=seed)
    if one_hot:
        env = DiscreteOneHotWrapper(env)
    if monitor_dir is not None:
        env = Monitor(env, monitor_dir)
    return env


def describe_environment(env: gym.Env) -> None:
    print("Observation space:", env.observation_space)
    print("Action space:", env.action_space)
    if isinstance(env.observation_space, spaces.Discrete):
        print("Number of states:", env.observation_space.n)
    if isinstance(env.action_space, spaces.Discrete):
        print("Number of actions:", env.action_space.n)


env_preview = make_frozenlake_env(seed=SEED)
describe_environment(env_preview)
env_preview.close()

## Tabular Q-learning implementation

We now implement a vanilla Q-learning loop with an exponentially decaying $\varepsilon$-greedy exploration schedule. Besides episodic returns we also record the mean absolute change in the Q-table per episode so that we can visualise convergence.

In [ ]:
def q_learning(
    env: gym.Env,
    episodes: int = 5000,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon_start: float = 1.0,
    epsilon_min: float = 0.05,
    epsilon_decay: float = 0.995,
) -> Tuple[np.ndarray, Dict[str, List[float]]]:
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    q_table = np.zeros((n_states, n_actions), dtype=np.float64)

    rewards: List[float] = []
    epsilons: List[float] = []
    q_changes: List[float] = []

    epsilon = epsilon_start
    for episode in range(episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0.0
        delta_sum = 0.0
        steps = 0

        while not done:
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(q_table[state]))

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_action = int(np.argmax(q_table[next_state]))
            td_target = reward + gamma * q_table[next_state, best_next_action]
            td_error = td_target - q_table[state, action]
            q_table[state, action] += alpha * td_error

            delta_sum += abs(td_error)
            total_reward += reward
            state = next_state
            steps += 1

        rewards.append(total_reward)
        epsilons.append(epsilon)
        q_changes.append(delta_sum / max(1, steps))
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

    metrics = {
        "episode_rewards": rewards,
        "epsilon": epsilons,
        "q_deltas": q_changes,
    }
    return q_table, metrics


tabular_env = make_frozenlake_env(seed=SEED)
q_table, tabular_metrics = q_learning(tabular_env)
tabular_env.close()

In [ ]:
rewards = np.asarray(tabular_metrics["episode_rewards"], dtype=np.float64)
window = 100
moving_avg = np.convolve(rewards, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(rewards, alpha=0.3, label="Episode reward")
axes[0].plot(np.arange(window - 1, len(rewards)), moving_avg, label=f"{window}-episode moving average")
axes[0].set_title("Tabular Q-learning rewards")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Return")
axes[0].legend()

axes[1].plot(tabular_metrics["epsilon"])
axes[1].set_title("Exploration schedule")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Epsilon")

axes[2].plot(tabular_metrics["q_deltas"])
axes[2].set_title("Mean absolute Q-value change")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("|ΔQ|")
plt.tight_layout()
plt.show()

state_values = np.max(q_table, axis=1).reshape(4, 4)
plt.figure(figsize=(5, 4))
plt.imshow(state_values, cmap="viridis")
plt.colorbar(label="State value (max_a Q)")
plt.title("Learned state values")
plt.xticks(range(4))
plt.yticks(range(4))
plt.show()

The tabular agent steadily improves over time despite the environment's stochastic transitions. The mean absolute Q-value changes taper off as the greedy policy begins to stabilise, and the learned state-value heatmap highlights the desirable goal state in the lower-right corner of the grid.

## Deep Q-learning with Stable-Baselines3

Next we switch to function approximation. Each algorithm receives the one-hot encoded observations and interacts with identical training and evaluation environments. We evaluate every 2,000 steps to monitor learning progress.

In [ ]:
class EvaluationLogger(BaseCallback):
    def __init__(self, eval_env: DummyVecEnv, eval_freq: int = 2000, n_eval_episodes: int = 10, verbose: int = 0):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.timesteps: List[int] = []
        self.mean_rewards: List[float] = []
        self.std_rewards: List[float] = []

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            mean_reward, std_reward = evaluate_policy(
                self.model,
                self.eval_env,
                n_eval_episodes=self.n_eval_episodes,
                deterministic=False,
            )
            self.timesteps.append(self.num_timesteps)
            self.mean_rewards.append(mean_reward)
            self.std_rewards.append(std_reward)
        return True

In [ ]:
def make_sb3_vec_env(seed: int) -> DummyVecEnv:
    def _init() -> gym.Env:
        env = make_frozenlake_env(seed=seed, one_hot=True)
        return env

    return DummyVecEnv([_init])


TOTAL_TIMESTEPS = 20000
EVAL_FREQ = 2000
N_EVAL_EPISODES = 15

algorithms = [
    ("DQN", DQN, dict(
        learning_rate=1e-3,
        buffer_size=50_000,
        exploration_fraction=0.3,
        exploration_final_eps=0.05,
        target_update_interval=500,
        train_freq=4,
        gradient_steps=1,
    )),
    ("QR-DQN", QRDQN, dict(
        learning_rate=5e-4,
        buffer_size=50_000,
        exploration_fraction=0.3,
        exploration_final_eps=0.05,
        target_update_interval=500,
        train_freq=4,
        gradient_steps=1,
        n_quantiles=51,
    )),
    ("C51", C51, dict(
        learning_rate=5e-4,
        buffer_size=50_000,
        exploration_fraction=0.3,
        exploration_final_eps=0.05,
        target_update_interval=500,
        train_freq=4,
        gradient_steps=1,
    )),
]

sb3_histories: Dict[str, Dict[str, List[float]]] = {}
sb3_final_scores: Dict[str, Tuple[float, float]] = {}

for name, algo_cls, hyperparams in algorithms:
    train_vec_env = make_sb3_vec_env(SEED)
    eval_vec_env = make_sb3_vec_env(SEED + 100)
    callback = EvaluationLogger(
        eval_vec_env,
        eval_freq=EVAL_FREQ,
        n_eval_episodes=N_EVAL_EPISODES,
    )
    model = algo_cls(
        "MlpPolicy",
        train_vec_env,
        gamma=0.99,
        verbose=0,
        seed=SEED,
        **hyperparams,
    )
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback, progress_bar=False)
    mean_reward, std_reward = evaluate_policy(
        model,
        eval_vec_env,
        n_eval_episodes=50,
        deterministic=False,
    )
    sb3_histories[name] = {
        "timesteps": callback.timesteps,
        "mean_rewards": callback.mean_rewards,
        "std_rewards": callback.std_rewards,
    }
    sb3_final_scores[name] = (mean_reward, std_reward)
    train_vec_env.close()
    eval_vec_env.close()

In [ ]:
plt.figure(figsize=(10, 5))
for name, history in sb3_histories.items():
    if history["timesteps"]:
        plt.plot(history["timesteps"], history["mean_rewards"], marker="o", label=name)
plt.title("Evaluation rewards for Stable-Baselines3 agents")
plt.xlabel("Training timesteps")
plt.ylabel(f"Mean reward over {N_EVAL_EPISODES} episodes")
plt.legend()
plt.grid(True)
plt.show()

print("Final evaluation over 50 episodes:")
for name, (mean_reward, std_reward) in sb3_final_scores.items():
    print(f"{name:>6s}: mean={mean_reward:.3f}, std={std_reward:.3f}")

Although all three agents eventually learn strong policies, the distributional variants (`QR-DQN` and `C51`) typically achieve more stable evaluation curves on this stochastic task. Their return distributions enable richer credit assignment compared to the vanilla DQN baseline.

## Comparative discussion

* **Sample efficiency:** The tabular method converges within a few thousand episodes because it stores exact state-action values, while neural agents require replay buffers and longer training horizons.
* **Exploration:** A decaying $\varepsilon$-greedy policy suffices for the small state space, but the neural agents benefit from SB3's built-in exploration schedules that balance replay stability and coverage.
* **Representation power:** Once the state space grows or observations become continuous, the tabular approach breaks down, whereas neural Q-learning scales by approximating the value function.

Taken together, the experiments illustrate how classical Q-learning ideas map onto modern deep RL implementations while highlighting the trade-offs introduced by function approximation.